In [2]:
import warnings
warnings.filterwarnings('ignore')
import hoomd
import datetime
import gsd
import matplotlib.pyplot as plt
import numpy as np
import gsd.hoomd
from flowermd.base import Pack,Lattice, Simulation
from flowermd.library import EllipsoidForcefield, EllipsoidChain
from flowermd.utils import get_target_box_number_density
from flowermd.utils.constraints import create_rigid_ellipsoid_chain
import unyt as u
import hoomd
import os

In [ ]:
# putting all configurable parameters into a dictionary makes parameters.txt automation easier
dt = 0.0001
parameters = {
    "lpar": 1.0,
    "lperp": 0.5,
    "num_mols": 128,
    "bead_mass": 1.0,
    "lengths": 1,
    "epsilon": 1.0,
    "r_cut": 3.0,
    "dt": dt,

    # packiing parameters
    "density_initial": 0.1*u.Unit("nm**-3"),
    "packing_expand_factor": 6,
    "pack_edge": 2,
    "pack_overlap": 1,
    "pack_fix_orientation": True,

    # shrinking parameters
    "density_final": 0.3*u.Unit("nm**-3"),
    "shrink_steps": 1e6,
    "shrink_kT": 1.0,
    "shrink_tau_kt": 5*dt,
    "shrink_period": 10,
    "shrink_thermalize_particles": False,

    # static sim parameters
    "static_steps": 2e5,
    "static_kT": 1.0,
    "static_tau_kt": 5*dt,
}

ellipsoid_chain = EllipsoidChain(
    lengths=parameters['lengths'],
    num_mols=parameters['num_mols'],
    lpar=parameters['lpar'],
    bead_mass=parameters['bead_mass'],
)
ff = EllipsoidForcefield(
    epsilon=parameters['epsilon'],
    lpar=parameters['lpar'],
    lperp=parameters['lperp'],
    r_cut=parameters['r_cut'],
)
system = Pack(
    molecules=ellipsoid_chain,
    density=parameters['density_initial'],
    packing_expand_factor=parameters['packing_expand_factor'],
    edge=parameters['pack_edge'],
    overlap=parameters['pack_overlap'],
    fix_orientation=parameters['pack_fix_orientation'],
)

time_string = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
output_dir = 'logs/' + time_string + '/'
try:
    os.makedirs(output_dir)
except OSError as error:
    print('failed to create directory ' + output_dir)

with open(output_dir + 'parameters.txt', 'a') as param_file:
    for key, val in parameters.items():
        param_file.write(str(key) + ': ' + str(val) + '\n')

GSD_FILE_PATH = output_dir + 'trajectory.gsd'
LOG_FILE_PATH = output_dir + 'log.txt'

rigid_frame, rigid = create_rigid_ellipsoid_chain(
    system.hoomd_snapshot
)
ellipsoid_sim = Simulation(
    initial_state=rigid_frame,
    forcefield=ff.hoomd_forces,
    constraint=rigid,
    gsd_write_freq=int(8e3),
    gsd_file_name=GSD_FILE_PATH,
    log_write_freq=int(1e4),
    log_file_name=LOG_FILE_PATH,
    dt=parameters['dt']
)

target_box = get_target_box_number_density(density=parameters['density_final'], n_beads=parameters['num_mols'])
ellipsoid_sim.run_update_volume(
    final_box_lengths=target_box,
    kT=parameters['shrink_kT'],
    n_steps=parameters['shrink_steps'],
    tau_kt=parameters['shrink_tau_kt'],
    period=parameters['shrink_period'],
    thermalize_particles=parameters['shrink_thermalize_particles']
)
print("shrink finished")
ellipsoid_sim.run_NVT(
   n_steps=parameters['static_steps'],
   kT=parameters['static_kT'],
   tau_kt=parameters['static_tau_kt'],
)
print("simulation finished")
#ellipsoid_sim.save_restart_gsd("restart.gsd")
ellipsoid_sim.flush_writers()
#ellipsoid_sim.save_simulation("sim.pickle")

Initializing simulation state from a gsd.hoomd.Frame.
Step 9000 of 1000000; TPS: 7550.43; ETA: 2.2 minutes
Step 18000 of 1000000; TPS: 8446.48; ETA: 1.9 minutes
Step 27000 of 1000000; TPS: 8582.82; ETA: 1.9 minutes
Step 36000 of 1000000; TPS: 8684.74; ETA: 1.8 minutes
Step 45000 of 1000000; TPS: 8746.56; ETA: 1.8 minutes
Step 54000 of 1000000; TPS: 8868.83; ETA: 1.8 minutes
Step 63000 of 1000000; TPS: 8958.83; ETA: 1.7 minutes
Step 72000 of 1000000; TPS: 8972.1; ETA: 1.7 minutes
Step 81000 of 1000000; TPS: 9048.02; ETA: 1.7 minutes
Step 90000 of 1000000; TPS: 9125.94; ETA: 1.7 minutes
Step 99000 of 1000000; TPS: 9201.43; ETA: 1.6 minutes
Step 108000 of 1000000; TPS: 9256.6; ETA: 1.6 minutes
Step 117000 of 1000000; TPS: 9306.03; ETA: 1.6 minutes
Step 126000 of 1000000; TPS: 9334.22; ETA: 1.6 minutes
Step 135000 of 1000000; TPS: 9367.39; ETA: 1.5 minutes
Step 144000 of 1000000; TPS: 9404.16; ETA: 1.5 minutes
Step 153000 of 1000000; TPS: 9433.29; ETA: 1.5 minutes
Step 162000 of 1000000; T

In [ ]:
def ellipsoid_gsd(gsd_file, new_file, ellipsoid_types, lpar, lperp):
    """Add needed information to GSD file to visualize ellipsoids.

    Saves a new GSD file with lpar and lperp values populated
    for each particle. Ovito can be used to visualize the new GSD file.

    Parameters
    ----------
    gsd_file : str
        Path to the original GSD file containing trajectory information
    new_file : str
        Path and filename of the new GSD file
    ellipsoid_types : str or list of str
        The particle types (i.e. names) of particles to be drawn
        as ellipsoids.
    lpar : float
        Value of lpar of the ellipsoids
    lperp : float
        Value of lperp of the ellipsoids

    """
    with gsd.hoomd.open(new_file, "w") as new_t:
        with gsd.hoomd.open(gsd_file) as old_t:
            for snap in old_t:
                shape_dicts_list = []
                for ptype in snap.particles.types:
                    if ptype == ellipsoid_types or ptype in ellipsoid_types:
                        shapes_dict = {
                            "type": "Ellipsoid",
                            "a": lpar,
                            "b": lperp,
                            "c": lperp,
                        }
                    else:
                        shapes_dict = {"type": "Sphere", "diameter": 0.001}
                    shape_dicts_list.append(shapes_dict)
                snap.particles.type_shapes = shape_dicts_list
                snap.validate()
                new_t.append(snap)

ellipsoid_gsd(
    gsd_file=GSD_FILE_PATH,
    new_file=GSD_FILE_PATH.replace('trajectory.gsd', 'ovito-trajectory.gsd'),
    ellipsoid_types='R',
    lpar=parameters['lpar'],
    lperp=parameters['lperp'],
)

In [ ]:
log = np.genfromtxt(LOG_FILE_PATH, names=True)
timestep = log["flowermdbasesimulationSimulationtimestep"]
potential_energy = log["mdcomputeThermodynamicQuantitiespotential_energy"]
kinetic_energy = log["mdcomputeThermodynamicQuantitieskinetic_energy"]
volume = log["mdcomputeThermodynamicQuantitiesvolume"]
temp = log["mdcomputeThermodynamicQuantitieskinetic_temperature"]

plt.plot(timestep, potential_energy)
plt.title('Potential Energy vs. Time')
plt.xlabel('Timestep')
plt.ylabel('Potential Energy')
plt.savefig(output_dir + 'potential-energy.png')
plt.close()

plt.plot(timestep, kinetic_energy)
plt.title('Kinetic Energy vs. Time')
plt.xlabel('Timestep')
plt.ylabel('Kinetic Energy')
plt.savefig(output_dir + 'kinetic-energy.png')
plt.close()

plt.plot(timestep, volume)
plt.title('Volume vs. Time')
plt.xlabel('Timestep')
plt.ylabel('Volume')
plt.savefig(output_dir + 'volume.png')
plt.close()

plt.plot(timestep, temp)
plt.title('Temperature vs. Time')
plt.xlabel('Timestep')
plt.ylabel('Temperature')
plt.savefig(output_dir + 'temp.png')
plt.close()

In [68]:
potential_energy

array([6.1928100e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 6.5022000e-01,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       2.5489900e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e

In [72]:
volume

array([8.6400000e+05, 8.3969700e+05, 8.1583000e+05, 7.9242000e+05,
       7.6946200e+05, 7.4695200e+05, 7.2488500e+05, 7.0325700e+05,
       6.8206300e+05, 6.6130000e+05, 6.4096200e+05, 6.2104600e+05,
       6.0154700e+05, 5.8246000e+05, 5.6378100e+05, 5.4550600e+05,
       5.2763000e+05, 5.1014900e+05, 4.9305800e+05, 4.7635400e+05,
       4.6003100e+05, 4.4408500e+05, 4.2851200e+05, 4.1330800e+05,
       3.9846700e+05, 3.8398600e+05, 3.6986000e+05, 3.5608500e+05,
       3.4265700e+05, 3.2957000e+05, 3.1682100e+05, 3.0440400e+05,
       2.9231700e+05, 2.8055400e+05, 2.6911000e+05, 2.5798300e+05,
       2.4716600e+05, 2.3665600e+05, 2.2644800e+05, 2.1653800e+05,
       2.0692200e+05, 1.9759400e+05, 1.8855100e+05, 1.7978800e+05,
       1.7130100e+05, 1.6308600e+05, 1.5513700e+05, 1.4745100e+05,
       1.4002300e+05, 1.3284900e+05, 1.2592400e+05, 1.1924400e+05,
       1.1280500e+05, 1.0660200e+05, 1.0063000e+05, 9.4885800e+04,
       8.9364300e+04, 8.4061300e+04, 7.8972400e+04, 7.4093100e

In [111]:
foo = 7*u.Unit("nm**-3")
print(foo)

7 nm**(-3)
